In [25]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
from pyathena import connect
from geopy.geocoders import Nominatim
from time import sleep
import os
from sqlalchemy import create_engine

# Load predictions data from S3

In [6]:
BUCKET = 'vegetation-risk-ml'
df = pd.read_csv(f's3://{BUCKET}/ml/predictions/test_predictions.csv')
print(df.shape)
df.head()

(243820, 17)


,dia,ht,slope,regional_drybiot,fire_recurrence,avg_temp,avg_rain,avg_wind,fuel_moisture_risk,log_fire_size,fire_month_sin,fire_month_cos,lat,lon,countycd,priority,risk_score
0,39.4,98.0,55.0,9499.979096,0.693147,29.156305,0.002346,11.024633,29.088063,3.044522,-0.5,-0.866025,39.526948,-122.672546,21,High,100.0
1,30.0,108.0,55.0,6222.471639,0.693147,29.156305,0.002346,11.024633,29.088063,3.044522,-0.5,-0.866025,39.526948,-122.672546,21,High,100.0
2,16.6,92.0,55.0,1681.913042,0.693147,29.156305,0.002346,11.024633,29.088063,3.044522,-0.5,-0.866025,39.526948,-122.672546,21,High,100.0
3,24.0,65.0,53.0,2432.080740,0.693147,29.156305,0.002346,11.024633,29.088063,3.044522,-0.5,-0.866025,39.526948,-122.672546,21,High,100.0
4,24.5,95.0,53.0,3186.055238,0.693147,29.156305,0.002346,11.024633,29.088063,3.044522,-0.5,-0.866025,39.526948,-122.672546,21,High,100.0


In [7]:
# Check if predictions vary by location
print(df.groupby('countycd')['priority'].value_counts())

# Check if risk score actually varies
print(df.groupby('priority')['risk_score'].describe())

# Check feature correlation with priority
print(df.groupby('priority')[['slope', 'avg_temp', 'avg_rain', 'fuel_moisture_risk']].mean())

countycd  priority
1         Medium       504
          High         161
          Low          154
3         Low         1843
          Medium      1617
                      ... 
113       High         106
          Low           30
115       Low         1221
          Medium      1076
          High         546
Name: count, Length: 159, dtype: int64
            count       mean       std    min    25%    50%    75%     max
priority                                                                  
High      81078.0  98.416790  4.149821  51.66  99.40  99.95  99.99  100.00
Low       81168.0   1.421213  3.866031   0.00   0.01   0.06   0.52   29.88
Medium    81574.0  50.205679  5.849587  25.06  49.59  50.05  50.67   74.98
              slope   avg_temp  avg_rain  fuel_moisture_risk
priority                                                    
High      33.707442  25.758639  0.172194           23.369136
Low       33.845690  17.337704  1.239548            9.840316
Medium    32.163937  21.58

The model predictions are validated by consistent environmental patterns across risk categories,  High priority areas exhibit significantly higher temperatures (25.8°C), lower rainfall (0.17), and greater fuel moisture risk (23.4) compared to Low priority areas, confirming that the model captures meaningful ecological relationships rather than arbitrary classifications.The spatial distribution of risk predictions varies meaningfully across California counties, with risk scores clearly separated, High risk averaging 98.4, Medium averaging 50.2, and Low averaging 1.4 showing that the model's 98.8% accuracy reflects genuine predictive power and that the results are reliable for operational vegetation risk assessment.

# Add County Name

In [8]:
# Mapping of county codes to names based on California county codes
county_map = {
    1: "Alameda", 3: "Alpine", 5: "Amador", 7: "Butte", 9: "Calaveras",
    11: "Colusa", 13: "Contra Costa", 15: "Del Norte", 17: "El Dorado",
    19: "Fresno", 21: "Glenn", 23: "Humboldt", 25: "Imperial",
    27: "Inyo", 29: "Kern", 31: "Kings", 33: "Lake", 35: "Lassen",
    37: "Los Angeles", 39: "Madera", 41: "Marin", 43: "Mariposa",
    45: "Mendocino", 47: "Merced", 49: "Modoc", 51: "Mono",
    53: "Monterey", 55: "Napa", 57: "Nevada", 59: "Orange",
    61: "Placer", 63: "Plumas", 65: "Riverside", 67: "Sacramento",
    69: "San Benito", 71: "San Bernardino", 73: "San Diego",
    75: "San Francisco", 77: "San Joaquin", 79: "San Luis Obispo",
    81: "San Mateo", 83: "Santa Barbara", 85: "Santa Clara",
    87: "Santa Cruz", 89: "Shasta", 91: "Sierra", 93: "Siskiyou",
    95: "Solano", 97: "Sonoma", 99: "Stanislaus",
    101: "Sutter", 103: "Tehama", 105: "Trinity",
    107: "Tulare", 109: "Tuolumne", 111: "Ventura",
    113: "Yolo", 115: "Yuba"
}
#apply mapping to create county_name column
df['county_name'] = df['countycd'].map(county_map)

print("Missing county names:", df['county_name'].isnull().sum())

Missing county names: 0


# Add city using geopy

In [10]:
geolocator = Nominatim(user_agent="geo_app")

def get_city(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en', timeout=10)
        address = location.raw.get('address', {})
        return address.get('city') or \
               address.get('town') or \
               address.get('village') or \
               address.get('county')
    except:
        return None

# Use unique locations to minimize API calls and avoid blocking
unique_locations = df[['lat', 'lon']].drop_duplicates().reset_index(drop=True)

cities = []
for i, row in unique_locations.iterrows():
    city = get_city(row['lat'], row['lon'])
    cities.append(city)
    sleep(1)
    if i % 100 == 0:
        print(f"Processed {i} rows")

unique_locations['city'] = cities

# Merge back using correct column names
df = df.merge(unique_locations, on=['lat', 'lon'], how='left')

# Fill missing
df['city'] = df['city'].fillna('Unknown')

Processed 0 rows
Processed 100 rows
Processed 200 rows
Processed 300 rows
Processed 400 rows
Processed 500 rows
Processed 600 rows
Processed 700 rows
Processed 800 rows
Processed 900 rows
Processed 1000 rows
Processed 1100 rows
Processed 1200 rows
Processed 1300 rows
Processed 1400 rows
Processed 1500 rows
Processed 1600 rows
Processed 1700 rows
Processed 1800 rows
Processed 1900 rows
Processed 2000 rows
Processed 2100 rows
Processed 2200 rows
Processed 2300 rows
Processed 2400 rows
Processed 2500 rows
Processed 2600 rows
Processed 2700 rows
Processed 2800 rows
Processed 2900 rows
Processed 3000 rows
Processed 3100 rows
Processed 3200 rows
Processed 3300 rows
Processed 3400 rows
Processed 3500 rows
Processed 3600 rows
Processed 3700 rows
Processed 3800 rows
Processed 3900 rows
Processed 4000 rows
Processed 4100 rows
Processed 4200 rows
Processed 4300 rows
Processed 4400 rows
Processed 4500 rows
Processed 4600 rows
Processed 4700 rows
Processed 4800 rows
Processed 4900 rows
Processed 50

In [11]:
# Rename for map visuals
df.rename(columns={'lat': 'latitude','lon': 'longitude'}, inplace=True)

# Drop bad rows
df = df.dropna(subset=['latitude', 'longitude', 'risk_score','county_name'])

print("Cleaned:", df.shape)

Cleaned: (243820, 19)


# Dashboard Features

In [12]:
# Remove null risk
df = df.dropna(subset=["risk_score"])

# Risk levels
df["risk_level"] = pd.cut(
    df["risk_score"],
    bins=[0, 30, 60, 100],
    labels=["Low Priority", "Medium Priority", "High Priority"],
    include_lowest=True
).astype(str)

# KPI fields
df["trimming_priority_score"] = df["risk_score"].round(1)

# handle null biomass by filling with 0 (assuming no data means no biomass)
df["biomass_tons_ha"] = df["regional_drybiot"].fillna(0)

# High risk flag
df["is_high_risk"] = (df["risk_score"] > 60).astype(int)

print("Feature engineering done")

Feature engineering done


In [13]:

# Validate required columns for dashboard

required_cols = [
    "countycd", "county_name", "city", "latitude", "longitude", "risk_score",
    "trimming_priority_score", "risk_level", "biomass_tons_ha", "is_high_risk",
    "fuel_moisture_risk", "fire_recurrence", "log_fire_size",
    "avg_temp", "avg_rain", "avg_wind", "slope"
]
# Check for missing columns
missing = [c for c in required_cols if c not in df.columns]
# If any required columns are missing, raise an error
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("All required columns present")

All required columns present


# Save to S3

In [17]:

# Plot-level data to s3 fo
df[required_cols].to_csv("s3://vegetation-risk-ml/dashboard/plots/predictions_clean.csv",index=False)

# Also save a sample locally for quick testing
os.makedirs("dashboard", exist_ok=True)
df[required_cols].sample(5000).to_csv("dashboard/sample_predictions.csv", index=False)


In [18]:
df[required_cols].head()

,countycd,county_name,city,latitude,longitude,risk_score,trimming_priority_score,risk_level,biomass_tons_ha,is_high_risk,fuel_moisture_risk,fire_recurrence,log_fire_size,avg_temp,avg_rain,avg_wind,slope
0,21,Glenn,Glenn County,39.526948,-122.672546,100.0,100.0,High Priority,9499.979096,1,29.088063,0.693147,3.044522,29.156305,0.002346,11.024633,55.0
1,21,Glenn,Glenn County,39.526948,-122.672546,100.0,100.0,High Priority,6222.471639,1,29.088063,0.693147,3.044522,29.156305,0.002346,11.024633,55.0
2,21,Glenn,Glenn County,39.526948,-122.672546,100.0,100.0,High Priority,1681.913042,1,29.088063,0.693147,3.044522,29.156305,0.002346,11.024633,55.0
3,21,Glenn,Glenn County,39.526948,-122.672546,100.0,100.0,High Priority,2432.080740,1,29.088063,0.693147,3.044522,29.156305,0.002346,11.024633,53.0
4,21,Glenn,Glenn County,39.526948,-122.672546,100.0,100.0,High Priority,3186.055238,1,29.088063,0.693147,3.044522,29.156305,0.002346,11.024633,53.0


In [19]:

# County summary (for charts)
county_summary = df.groupby(["county_name", "countycd", "risk_level"]).agg(
    avg_risk_score=("risk_score", "mean"),
    total_plots=("risk_score", "count"),
    high_risk_plots=("is_high_risk", "sum"),
    avg_biomass=("biomass_tons_ha", "mean"),
    avg_moisture=("fuel_moisture_risk", "mean"),
    avg_fire_recurrence=("fire_recurrence", "mean"),
    avg_fire_size=("log_fire_size", "mean"),
    avg_temp=("avg_temp", "mean"),
    avg_rain=("avg_rain", "mean"),
    avg_wind=("avg_wind","mean"),   
    avg_slope=("slope","mean"),  
    avg_lat=("latitude", "mean"),
    avg_lon=("longitude", "mean")
).reset_index()

# Save county summary to S3 for dashboard
county_summary.to_csv("s3://vegetation-risk-ml/dashboard/county/county_summary.csv",index=False)
print("Files saved to S3")

Files saved to S3


# Connect to Athena

In [20]:
# Connect to Athena for dashboard queries
s3_staging_dir = "s3://vegetation-risk-ml/athena-results/"

conn = connect(s3_staging_dir=s3_staging_dir,region_name="us-east-1")
cursor = conn.cursor()

print("Connected to Athena")

Connected to Athena


# Create Table

In [21]:

# Create external table in Athena for dashboard queries
cursor.execute("DROP TABLE IF EXISTS vegetation_ml.predictions_clean")

create_dbtable = """
CREATE EXTERNAL TABLE vegetation_ml.predictions_clean (
    countycd INT,
    county_name STRING,
    city STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    risk_score DOUBLE,
    trimming_priority_score DOUBLE,
    risk_level STRING,
    biomass_tons_ha DOUBLE,
    is_high_risk INT,
    fuel_moisture_risk DOUBLE,
    fire_recurrence DOUBLE,
    log_fire_size DOUBLE,
    avg_temp DOUBLE,
    avg_rain DOUBLE,
    avg_wind DOUBLE,
    slope DOUBLE
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
LOCATION 's3://vegetation-risk-ml/dashboard/plots/'
TBLPROPERTIES ("skip.header.line.count"="1")
"""

cursor.execute(create_dbtable)

print("Table for dashboard created")

Table for dashboard created


# Create Views in Athena

In [23]:

# Create dashboard view
create_view = """
CREATE OR REPLACE VIEW vegetation_ml.dashboard_view AS
SELECT
    county_name,
    countycd,
    risk_level,
    ROUND(AVG(risk_score),1) AS avg_risk_score,
    COUNT(*) AS total_plots,
    SUM(is_high_risk) AS high_risk_plots,
    ROUND(AVG(biomass_tons_ha),1) AS avg_biomass,
    ROUND(AVG(fuel_moisture_risk),1) AS avg_moisture,
    ROUND(AVG(fire_recurrence),1) AS avg_fire_recurrence,
    ROUND(AVG(log_fire_size),1) AS avg_fire_size,
    ROUND(AVG(avg_temp),1) AS avg_temp,
    ROUND(AVG(avg_rain),1) AS avg_rain,
    ROUND(AVG(avg_wind),1) AS avg_wind,
    ROUND(AVG(slope),1) AS avg_slope,
    AVG(latitude) AS avg_lat,
    AVG(longitude) AS avg_lon
FROM vegetation_ml.predictions_clean
GROUP BY county_name, countycd, risk_level
"""

cursor.execute(create_view)

print("Views for dashboard created")

Views for dashboard created


# Validate Query

In [26]:
# Test query
engine = create_engine(
    "awsathena+rest://@athena.us-east-1.amazonaws.com:443/vegetation_ml"
    "?s3_staging_dir=s3://vegetation-risk-ml/athena-results/"
)
test = pd.read_sql("SELECT * FROM vegetation_ml.dashboard_view LIMIT 10", engine)
print(test.head())
print("Dashboard prep complete")

  county_name  countycd     risk_level  avg_risk_score  total_plots  \
0      Shasta        89  High Priority            98.0        12378   
1   Riverside        65  High Priority            95.8          219   
2       Modoc        49  High Priority            93.3         1956   
3      Fresno        19  High Priority            98.0         7796   
4   San Mateo        81  High Priority            93.4          344   

   high_risk_plots  avg_biomass  avg_moisture  avg_fire_recurrence  \
0            12378       2129.3          26.5                  0.7   
1              219       3874.5          22.6                  0.7   
2             1956       3160.5          18.8                  0.7   
3             7796       4208.5          23.9                  0.7   
4              344       9860.4          14.6                  0.7   

   avg_fire_size  avg_temp  avg_rain  avg_wind  avg_slope    avg_lat  \
0            6.4      28.6       0.1       8.5       35.5  40.791142   
1       